# 1. Dataset check — Bangla handedness paragraphs (42 writers)
Verifies every image against `SHA256SUMS.txt`, summarises the metadata and shows sample paragraphs.
Runs on CPU in about 2 minutes. No GPU needed.

In [ ]:
import os, sys, glob, shutil, subprocess, time, json
from pathlib import Path
# --- locate the uploaded Kaggle dataset (any slug) ---
KAGGLE = os.environ.get("KAGGLE_ROOT", "/kaggle")      # override only for local testing
hits = glob.glob(f"{KAGGLE}/input/**/dataset/metadata/images.csv", recursive=True)
assert hits, "Add the uploaded dataset to this notebook (Add data -> Your datasets)."
INPUT = Path(hits[0]).parents[2]
PROJECT = Path(f"{KAGGLE}/working/project")
(PROJECT / "results").mkdir(parents=True, exist_ok=True)
if not (PROJECT / "scripts").exists():
    shutil.copytree(INPUT / "code", PROJECT / "scripts")
if not (PROJECT / "dataset").exists():
    os.symlink(INPUT / "dataset", PROJECT / "dataset")
raw = PROJECT / "camera_originals/Right-Handed"; raw.mkdir(parents=True, exist_ok=True)
for f in ["P3", "P19"]:
    if not (raw / f).exists():
        os.symlink(INPUT / "extra/excluded_raw" / f, raw / f)
geo = PROJECT / "audit/geometric"; geo.mkdir(parents=True, exist_ok=True)
for f in ["geometric_pairs.csv.gz", "summary.json"]:                # dataset validation (Fig. 1f, audit numbers in the text)
    if not (geo / f).exists():
        os.symlink(INPUT / "extra/audit" / f, geo / f)
os.environ["TORCH_HOME"] = str(INPUT / "weights/torch")          # offline ImageNet weights
os.environ["MPLCONFIGDIR"] = "/tmp/mpl"
PY = sys.executable
JOBS = os.cpu_count() or 4
print("input:", INPUT, "| project:", PROJECT, "| cpus:", JOBS)
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

def run(args, log, done_file=None):
    """Run one pipeline script; skip if its main output already exists (lets you resume after a timeout)."""
    if done_file and Path(done_file).exists():
        print("skip (exists):", done_file); return
    t = time.time(); print("running:", " ".join(map(str, args)), flush=True)
    with open(log, "w") as fh:
        r = subprocess.run([PY, *map(str, args)], cwd=PROJECT, stdout=fh, stderr=subprocess.STDOUT)
    tail = Path(log).read_text().splitlines()[-15:]
    print("\n".join(l for l in tail if "resource_tracker" not in l and "KeyError" not in l))
    print(f"exit {r.returncode} after {time.time() - t:.0f}s"); assert r.returncode == 0, f"failed, see {log}"


In [ ]:
import hashlib, pandas as pd
from PIL import Image
D = PROJECT / "dataset"
m = pd.read_csv(D / "metadata/images.csv")
sums = dict(l.split("  ")[::-1] for l in (D / "SHA256SUMS.txt").read_text().strip().splitlines())
bad = [r.relative_path for r in m.itertuples()
       if hashlib.sha256((D / r.relative_path).read_bytes()).hexdigest() != sums[r.relative_path]]
print("images:", len(m), "| writers:", m.writer_id.nunique(), m.groupby("hand").writer_id.nunique().to_dict())
print("checksum mismatches:", len(bad))
print("sizes:", (m.width.astype(str) + "x" + m.height.astype(str)).value_counts().to_dict())
print("paragraphs per writer:", m.groupby("writer_id").size().unique())
print("truncated photos:", int(m.text_truncated_at_edge.notna().sum()))
display(pd.read_csv(D / "metadata/writers.csv").head(10))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, (w, k) in zip(axes.ravel(), [("L_p3", 1), ("L_p10", 1), ("R_p7", 1), ("R_p16", 1), ("L_p3", 7), ("L_p10", 7), ("R_p7", 7), ("R_p16", 7)]):
    r = m[(m.writer_id == w) & (m.paragraph_number == k)].iloc[0]
    ax.imshow(Image.open(D / r.relative_path)); ax.set_title(f"{w} ({r.hand}) paragraph {k}"); ax.axis("off")
plt.tight_layout()